# 👻  **Reto 2 - NLP CUDA NUMBA 🐦**

<img src="https://www.udistrital.edu.co/themes/custom/versh/images/default/preloader.png" align="left" width="192px" height="192px"/>
<img align="left" width="0" height="192px" hspace="10"/>

> Juan Felipe Rodríguez Galindo  - **COD. 20261595004**
<br></br>
[![Juan Felipe Rodriguez Galindo](https://img.shields.io/badge/Juferoga-github-br?style=flat-square)](https://github.com/Juferoga)
[![License](https://img.shields.io/badge/License-GPL_V.3-blue?style=flat-square)](https://www.gnu.org/licenses/gpl-3.0.html)


Computación paralela

Desarrrollo del reto 2 propuesto en la clase de Computación Paralela.

# Proyecto final: GPU NLP Pipeline con CUDA + Numba

**Punto 1.** Mediante comentarios explicar el código del presente Colab.

**Punto 2.** Ajustar el Pipeline para que el Corpus sea un documento considerablemente grande.

## Análisis masivo de texto usando GPU

Este notebook implementa un pipeline pedagógico de Procesamiento de Lenguaje Natural (NLP) usando **CUDA con Numba**. El reto consiste en acelerar tareas básicas de NLP sobre GPU:

1. Tokenización básica: caracteres → tokens  
2. Conteo de términos: Bag of Words  
3. Cálculo de TF-IDF  
4. N-gramas: bigramas  
5. Puntaje de sentimiento léxico  

> **Requisito:** en Google Colab activar GPU: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU`


## Resultados de aprendizaje

Al finalizar el notebook, el estudiante podrá convertir texto a una representación numérica, implementar kernels CUDA con Numba, construir Bag of Words, calcular TF-IDF, generar bigramas, calcular sentimiento léxico y comparar CPU vs GPU.


# 1. Verificación del entorno

In [3]:
# Sitema operativo
!uname -a

Linux 50da710fd4b4 6.6.159+ #1 SMP Thu Oct 22 03:07:54 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux


In [2]:
# Permite ver el driver de la tarjeta (Nvidia)
!nvidia-smi

Mon Oct 26 17:05:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# importamos librerias necesarias para el proyecto
import numpy as np # manejo de matrices
import math # Matemáticas
import time # Control y medición de tiempo
from numba import cuda # libreria para el uso de cuda desde el paquete numba

# Revisamos la versión de numpy del entorno
print("NumPy:", np.__version__)
# Se revisa si cuda está disponible, por si elegimos mal la máquina
print("CUDA disponible:", cuda.is_available())
if cuda.is_available():
    # GPU deberia ser la misma que al hacer smi-nvidia
    print("GPU actual:", cuda.get_current_device().name)


NumPy: 2.1.3
CUDA disponible: True
GPU actual: Tesla T4


# 2. Corpus de trabajo (Punto 2: corpus considerablemente grande)

El notebook original replicaba 10 frases 5000 veces. Eso da 50.000 "documentos", pero en realidad hay **solo 10 textos distintos** y unas 35 palabras de vocabulario: la GPU procesa datos redundantes y el resultado no se parece a un caso real.

Para el **Punto 2** el corpus se construye con **texto real y grande**: seis obras en español del Proyecto Gutenberg (dominio público), más de **1,2 millones de palabras**:

| ID Gutenberg | Obra |
|---|---|
| 2000 | *Don Quijote* — Cervantes |
| 17013 | *Fortunata y Jacinta* — Pérez Galdós |
| 24536 | *Los cuatro jinetes del Apocalipsis* — Blasco Ibáñez |
| 58221 | *La Odisea* (traducción) |
| 49836 | *Niebla* — Unamuno |
| 15115 | *Novelas y teatro* — Cervantes |

Pasos del preprocesamiento:

1. Descargar cada libro y quitar el encabezado/pie de licencia de Gutenberg.
2. **Normalizar**: minúsculas y quitar tildes (`á→a`, `ñ→n`). El kernel de tokenización solo reconoce ASCII `a-z0-9`, así que sin este paso las palabras con tilde se partirían en pedazos.
3. **Segmentar** el texto en "documentos" de `PALABRAS_POR_DOC` palabras (ventanas sin traslape), algo así como párrafos cortos.

Si no hay internet, se usa como respaldo el corpus sintético replicado del notebook original.

In [ ]:
import re # expresiones regulares para limpiar el texto
import unicodedata # para quitar tildes (normalización NFKD)
import urllib.request # descarga de los libros

# Libros en español del Proyecto Gutenberg (id, título)
LIBROS = [
    (2000, "Don Quijote"),
    (17013, "Fortunata y Jacinta"),
    (24536, "Los cuatro jinetes del Apocalipsis"),
    (58221, "La Odisea"),
    (49836, "Niebla"),
    (15115, "Novelas y teatro"),
]

# Cuántas palabras forman un "documento" del corpus
PALABRAS_POR_DOC = 20
# Tope opcional de documentos (None = usar todo el texto descargado)
MAX_DOCS = None

def descargar_libro(book_id):
    # Descarga el .txt plano del libro desde Gutenberg
    url = f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt"
    with urllib.request.urlopen(url, timeout=60) as resp:
        texto = resp.read().decode("utf-8", errors="ignore")
    # Gutenberg agrega licencia al inicio y al final, se recorta lo que está
    # entre las marcas "*** START OF ..." y "*** END OF ..."
    inicio = re.search(r"\*\*\* ?START OF.*?\*\*\*", texto)
    fin = re.search(r"\*\*\* ?END OF", texto)
    if inicio and fin:
        texto = texto[inicio.end():fin.start()]
    return texto

def normalizar(texto):
    # Minúsculas + NFKD separa la letra de su tilde ("á" -> "a" + "´")
    texto = unicodedata.normalize("NFKD", texto.lower())
    # Se eliminan las marcas de tilde y cualquier otro carácter no ASCII
    # (¿, ¡, «, —) se vuelve espacio para no pegar palabras vecinas
    texto = "".join(" " if ord(c) > 127 else c
                    for c in texto if not unicodedata.combining(c))
    # Colapsamos espacios, saltos de línea y tabulaciones
    return re.sub(r"\s+", " ", texto).strip()

def construir_corpus(libros, palabras_por_doc, max_docs=None):
    docs = []
    for book_id, titulo in libros:
        try:
            palabras = normalizar(descargar_libro(book_id)).split(" ")
        except Exception as e:
            print(f"  No se pudo descargar {titulo} ({book_id}): {e}")
            continue
        # Ventanas de palabras_por_doc palabras -> un documento cada una
        nuevos = [" ".join(palabras[k:k + palabras_por_doc])
                  for k in range(0, len(palabras), palabras_por_doc)]
        docs.extend(nuevos)
        print(f"  {titulo:<36} {len(palabras):>9,} palabras -> {len(nuevos):>7,} docs")
    return docs[:max_docs] if max_docs else docs

# Corpus del notebook original: se conserva como respaldo sin internet
base_corpus = [
    "La inteligencia artificial transforma la universidad",
    "La universidad mejora sus procesos con analitica",
    "Los estudiantes usan inteligencia artificial",
    "La analitica predictiva apoya la planeacion estrategica",
    "La universidad enfrenta desafios de innovacion",
    "La inteligencia artificial mejora la toma de decisiones",
    "Los datos permiten anticipar escenarios futuros",
    "La planeacion estrategica necesita datos e inteligencia",
    "La inteligencia artificial ayuda a resolver problemas urbanos",
    "La falta de datos genera riesgo en la toma de decisiones",
]

print("Descargando libros de Gutenberg...")
corpus = construir_corpus(LIBROS, PALABRAS_POR_DOC, MAX_DOCS)

if not corpus:
    # Respaldo: ampliación del corpus replicando los registros base
    REPETICIONES = 5000
    corpus = [normalizar(d) for d in base_corpus] * REPETICIONES
    print("Sin internet: se usa el corpus sintético replicado")

total_palabras = sum(len(d.split()) for d in corpus)
print()
print("Número de documentos:", f"{len(corpus):,}")
print("Total de palabras:   ", f"{total_palabras:,}")
print("Tamaño del texto:    ", f"{sum(len(d) for d in corpus) / 1e6:.2f} MB")
print("Vocabulario real (palabras distintas):", f"{len(set(' '.join(corpus).split())):,}")
print("Ejemplo:", corpus[1000 if len(corpus) > 1000 else 0])

# 3. Tokenización básica: caracteres → tokens

Tokenizar consiste en convertir un texto en unidades mínimas llamadas **tokens**. En este notebook, un token será una palabra.

Ejemplo:

```text
"La inteligencia artificial transforma la ciudad"
```

se convierte en:

```text
["la", "inteligencia", "artificial", "transforma", "la", "ciudad"]
```

CUDA no trabaja cómodamente con strings de Python. Por eso convertimos cada carácter en un número ASCII y luego un kernel CUDA recorre esos caracteres y construye tokens mediante un hash entero.


# 4. Preprocesamiento CPU: texto → matriz de caracteres

In [13]:
# Metodo para convertir el corpus a formato matricial, en este caso
# frases como "la inteligencia artificial transforma la universidad"
# se codifica y queda un arreglo de números [108, 97, 32, ..., 100]
# donde el [108, 97]="la" y así sucesivamente devolviendo el arreglo de
# números creados y su longitud teniendo como máximo de caracteres a evualar
# max_chars, si se pasa no se tiene en cuenta el resto de texto
def corpus_to_char_matrix(corpus, max_chars=160):

    # Matriz de ceros donde se guadaran los valores de los caracteres
    char_matrix = np.zeros((len(corpus), max_chars), dtype=np.int32)
    # Matriz de ceros donde se guadaran las longitudes de los docs del corpus
    doc_lengths = np.zeros(len(corpus), dtype=np.int32)

    for i, doc in enumerate(corpus):
        # Normalizamos poniendo todo en minuscula
        doc = doc.lower()
        # Secciona por caracteres y les pone su correspondiente codigo unicode
        # https://en.wikipedia.org/wiki/List_of_Unicode_characters
        encoded = [ord(c) for c in doc[:max_chars]]
        # Longitud por cada segmento codificado
        doc_lengths[i] = len(encoded)

        # Mio para saber que hace y que tiene
        if (i == 0):
          print(f"Del documento {i}, se evalua el \"{doc}\" y enc {encoded}")

        for j, code in enumerate(encoded):
            # guarda del encoded la matriz de caracteres
            char_matrix[i, j] = code

            # Mio para saber que hace y que tiene
            if (i == 0 and j == 0):
              print(f"En {i}, {j} podemos ver {char_matrix[i,j]} = {chr(code)}")
    # retorna las variables creadas
    return char_matrix, doc_lengths

# Definimos los caracteres máximos. Con el corpus grande cada documento tiene
# PALABRAS_POR_DOC=20 palabras (~120 caracteres en promedio), así que 320 deja
# margen para documentos con palabras largas sin desperdiciar mucha memoria
MAX_CHARS = 320
# Definimos los tokens máximos: no se usa aquí sino en el kernel 1, es el
# número de columnas de la matriz de hashes (máximo de tokens por documento)
MAX_TOKENS = 48

# ejecutamos la función previa, con el máximo de carateres a tener en cuenta
char_matrix, doc_lengths = corpus_to_char_matrix(corpus, MAX_CHARS)

# otras impresiones
print("Forma char_matrix:", char_matrix.shape)
print("Primer documento como códigos:", char_matrix[0, :50])
print("Longitud primer documento:", doc_lengths[0])
print("Documentos truncados por MAX_CHARS:", int(np.sum(np.array([len(d) for d in corpus]) > MAX_CHARS)))


# 5. Función auxiliar: hash de palabra en CPU

In [ ]:
import numpy as np

# Hash polinomial de una palabra (misma idea que String.hashCode de Java):
# h = c0*31^(n-1) + c1*31^(n-2) + ... + c(n-1). El módulo evita desbordar
# int32. La GPU debe usar EXACTAMENTE la misma fórmula, si no los hashes de
# la lista de sentimiento (calculados aquí) no coinciden con los del kernel.
def word_hash(word):

    h = 0
    MOD = 2_147_483_647  # máximo positivo para int32

    for c in word.lower():
        code_ascii = ord(c)
        # Solo cuentan letras a-z y dígitos 0-9 (igual que en el kernel)
        is_letter_or_digit = (97 <= code_ascii <= 122) or (48 <= code_ascii <= 57)

        if is_letter_or_digit:
            # Desplaza el hash acumulado (×31, primo) y suma el nuevo carácter
            h = (h * 31 + code_ascii) % MOD

    # Se devuelve como int32 para compararlo con los hashes de la GPU
    return np.int32(h)


print("Hash de inteligencia:", word_hash("inteligencia"))
print("Hash de artificial:", word_hash("artificial"))

# 6. Copiar datos a GPU

In [ ]:
# Copia host -> device (RAM -> VRAM) de la matriz de caracteres y las longitudes
d_chars = cuda.to_device(char_matrix)
d_lengths = cuda.to_device(doc_lengths)

# Salidas del kernel de tokenización: un hash por token (máx MAX_TOKENS por doc)
# y el número de tokens encontrados en cada documento
token_hashes = np.zeros((len(corpus), MAX_TOKENS), dtype=np.int32)
token_counts = np.zeros(len(corpus), dtype=np.int32)

d_token_hashes = cuda.to_device(token_hashes)
d_token_counts = cuda.to_device(token_counts)

# Configuración de lanzamiento 1D: un hilo por documento, 128 hilos por bloque
# (múltiplo de 32 = tamaño de warp). Los bloques se redondean hacia arriba
# para cubrir todos los documentos; los hilos sobrantes se descartan con un if.
threads = 128
blocks_docs = (len(corpus) + threads - 1) // threads

print("Bloques:", blocks_docs, "Hilos por bloque:", threads)
print(f"Memoria en GPU hasta ahora: {(d_chars.nbytes + d_token_hashes.nbytes) / 1e6:.1f} MB")

# 7. Kernel 1: Tokenización en GPU

Cada hilo CUDA procesa un documento. El kernel recorre caracteres, identifica letras/números, construye un hash por palabra, guarda tokens y registra cuántos tokens detectó.


> **Corrección importante.** En el kernel original el hash se calculaba como `h = h * 31 + c` **sin módulo**. Para palabras de más de ~6 letras el valor desborda y, al guardarlo en `int32`, queda truncado (por eso salía `inteligencia → -1923359874` en la GPU y `1509910231` en la CPU). Como las listas de sentimiento se codifican con `word_hash` (que sí aplica el módulo), **ninguna palabra larga coincidía** y el sentimiento de la GPU era incorrecto. Se agregó `% MOD` en el kernel para que GPU y CPU produzcan el mismo hash.

In [ ]:
# Módulo del hash, el mismo que usa word_hash en CPU (2^31 - 1)
HASH_MOD = 2_147_483_647

@cuda.jit
def tokenize_kernel(chars, lengths, token_hashes, token_counts, max_tokens):

    # Índice global del hilo = blockIdx.x * blockDim.x + threadIdx.x
    # Cada hilo se encarga de un documento completo
    doc_id = cuda.grid(1)

    # Los últimos bloques pueden tener hilos de más: no deben tocar memoria
    if doc_id < chars.shape[0]:

        # Hash acumulado de la palabra que se está leyendo
        h = 0

        # Posición donde se guardará el siguiente token del documento
        token_idx = 0

        # Indica si el carácter anterior pertenecía a una palabra
        in_token = False

        # Recorre solo los caracteres reales del documento (no el relleno de ceros)
        for j in range(lengths[doc_id]):

            # Código ASCII del carácter j del documento. int() lo promueve a
            # int64 para que h * 31 + c no desborde antes de aplicar el módulo
            c = int(chars[doc_id, j])

            # Una palabra está formada por letras a-z (97-122) y dígitos 0-9 (48-57)
            is_letter_or_digit = (c >= 97 and c <= 122) or (c >= 48 and c <= 57)

            if is_letter_or_digit:
                # Mismo hash polinomial que word_hash (con módulo, ver corrección)
                h = (h * 31 + c) % HASH_MOD

                # Estamos dentro de una palabra
                in_token = True

            else:
                # Un separador (espacio, coma, punto...) cierra la palabra actual:
                # se guarda su hash si aún hay espacio en la fila del documento
                if in_token and token_idx < max_tokens:
                    token_hashes[doc_id, token_idx] = h
                    token_idx += 1

                # Se reinicia el estado para la siguiente palabra
                h = 0
                in_token = False

        # Si el documento termina en una letra, la última palabra no tuvo
        # separador que la cerrara: se guarda aquí
        if in_token and token_idx < max_tokens:
            token_hashes[doc_id, token_idx] = h
            token_idx += 1

        # Número de tokens encontrados (lo usan todos los kernels siguientes)
        token_counts[doc_id] = token_idx

In [ ]:
# Lanzamiento del kernel: [número de bloques, hilos por bloque]
tokenize_kernel[blocks_docs, threads](
    d_chars,
    d_lengths,
    d_token_hashes,
    d_token_counts,
    MAX_TOKENS
)

# Los lanzamientos son asíncronos: se espera a que la GPU termine
cuda.synchronize()

# Copia device -> host para inspeccionar el resultado
token_counts_cpu = d_token_counts.copy_to_host()
token_hashes_cpu = d_token_hashes.copy_to_host()

print("Documento:", corpus[0])
print("Tokens detectados primer documento:", token_counts_cpu[0])
print("Hashes primer documento:", token_hashes_cpu[0, :token_counts_cpu[0]])
print("Tokens totales en el corpus:", f"{token_counts_cpu.sum():,}")
print("Documentos que llegaron al tope MAX_TOKENS:", int(np.sum(token_counts_cpu == MAX_TOKENS)))
# Comprobación de la corrección: el hash GPU de cada palabra debe ser igual al de CPU
primeras = corpus[0].split()[:5]
print("Hashes CPU (word_hash):", [int(word_hash(w)) for w in primeras], primeras)
print("Coinciden con la GPU:", [int(word_hash(w)) for w in primeras] == list(token_hashes_cpu[0, :5]))

# 8. Bag of Words

Bag of Words representa un documento como un vector de conteos. Como usamos hashing, cada token se asigna a una columna con `bucket = abs(hash) % vocab_size`.

> Con el corpus real el vocabulario tiene decenas de miles de palabras distintas y solo hay `VOCAB_SIZE = 2048` columnas, así que varias palabras comparten columna (**colisiones del hashing trick**). Es el precio de tener matrices de tamaño fijo en GPU; subir `VOCAB_SIZE` reduce colisiones a cambio de más memoria (`docs × VOCAB_SIZE × 4 bytes` por matriz).

In [ ]:
# Número de columnas de la matriz BoW (tamaño del "vocabulario" con hashing)
VOCAB_SIZE = 2048

# Matriz documentos × vocabulario en ceros (se llena con sumas atómicas)
bow = np.zeros((len(corpus), VOCAB_SIZE), dtype=np.int32)
d_bow = cuda.to_device(bow)

print(f"Matriz BoW: {bow.shape} -> {bow.nbytes / 1e6:.1f} MB en GPU")

# 9. Kernel 2: Construcción de Bag of Words

In [ ]:
@cuda.jit
def bow_kernel(token_hashes, token_counts, bow, vocab_size):

    # Un hilo por documento
    doc_id = cuda.grid(1)

    # Se descartan los hilos fuera de rango
    if doc_id < token_hashes.shape[0]:

        # Recorre los tokens válidos del documento
        for j in range(token_counts[doc_id]):

            # Hash del token j
            h = token_hashes[doc_id, j]

            # Columna del vocabulario donde cae el token (hashing trick)
            bucket = abs(h) % vocab_size

            # Suma 1 al conteo. Como cada hilo escribe solo en su propia fila,
            # un "+= 1" normal también serviría; el atómico garantiza
            # corrección aunque más de un hilo escribiera la misma celda
            cuda.atomic.add(bow, (doc_id, bucket), 1)

In [ ]:
# Un hilo por documento, misma configuración que la tokenización
bow_kernel[blocks_docs, threads](
    d_token_hashes,
    d_token_counts,
    d_bow,
    VOCAB_SIZE
)

cuda.synchronize()

# Se trae la matriz a CPU para revisar el primer documento
bow_cpu = d_bow.copy_to_host()
print("Términos no cero en primer documento:", np.count_nonzero(bow_cpu[0]))
# La suma de la fila debe ser igual al número de tokens del documento
print("Suma de conteos primer documento:", bow_cpu[0].sum(), "| tokens:", token_counts_cpu[0])
print("Columnas del vocabulario usadas en todo el corpus:", np.count_nonzero(bow_cpu.sum(axis=0)), "de", VOCAB_SIZE)

# 10. TF-IDF

**TF = ocurrencias / total de palabras** mide la frecuencia local de un término en un documento.
**IDF = log((1 + N) / (1 + DF)) + 1** mide qué tan raro o informativo es un término en el corpus (versión *suavizada*, la misma de scikit-learn: el `+1` evita dividir por cero y que un término presente en todos los documentos quede con peso 0).
**TF-IDF = TF × IDF**.

In [ ]:
# Matrices de trabajo. tf y tfidf se escriben completas en el kernel, por eso
# basta reservarlas en GPU sin copiar ceros (device_array). df sí debe
# empezar en cero porque se acumula con sumas atómicas.
d_tf = cuda.device_array((len(corpus), VOCAB_SIZE), dtype=np.float32)
d_df = cuda.to_device(np.zeros(VOCAB_SIZE, dtype=np.int32))
d_idf = cuda.device_array(VOCAB_SIZE, dtype=np.float32)
d_tfidf = cuda.device_array((len(corpus), VOCAB_SIZE), dtype=np.float32)

# Malla 2D: un hilo por celda (documento i, término j), bloques de 16×16 = 256 hilos
threads2d = (16, 16)
blocks2d = (
    (len(corpus) + threads2d[0] - 1) // threads2d[0],
    (VOCAB_SIZE + threads2d[1] - 1) // threads2d[1]
)

# Malla 1D sobre el vocabulario para el IDF (un hilo por término)
blocks_vocab = (VOCAB_SIZE + threads - 1) // threads

print("Malla 2D:", blocks2d, "bloques de", threads2d, "->", f"{blocks2d[0]*blocks2d[1]*256:,} hilos")

# 11. Kernel 3: Cálculo de TF

In [ ]:
@cuda.jit
def tf_kernel(bow, token_counts, tf):

    # Coordenadas 2D del hilo: i = documento, j = término
    i, j = cuda.grid(2)

    # Solo hilos dentro de la matriz
    if i < bow.shape[0] and j < bow.shape[1]:

        # TF = conteo del término / total de tokens del documento
        if token_counts[i] > 0:

            tf[i, j] = bow[i, j] / token_counts[i]
        else:
            # Documento vacío: se evita la división por cero
            tf[i, j] = 0.0

# 12. Kernel 4: Cálculo de DF

In [ ]:
@cuda.jit
def df_kernel(bow, df):

    # Un hilo por celda (documento i, término j)
    i, j = cuda.grid(2)

    if i < bow.shape[0] and j < bow.shape[1]:

        # DF[j] = número de documentos donde aparece el término j.
        # Muchos hilos (todos los documentos) escriben la misma posición df[j],
        # aquí el atómico SÍ es obligatorio para no perder incrementos
        if bow[i, j] > 0:
            cuda.atomic.add(df, j, 1)

# 13. Kernel 5: Cálculo de IDF

In [ ]:
@cuda.jit
def idf_kernel(df, idf, n_docs):

    # Un hilo por término del vocabulario
    j = cuda.grid(1)

    if j < df.size:

        # IDF suavizado: términos raros (df pequeño) obtienen un peso alto
        idf[j] = math.log((1.0 + n_docs) / (1.0 + df[j])) + 1.0

# 14. Kernel 6: Cálculo de TF-IDF

In [ ]:
@cuda.jit
def tfidf_kernel(tf, idf, tfidf):

    # Un hilo por celda (documento i, término j)
    i, j = cuda.grid(2)

    if i < tf.shape[0] and j < tf.shape[1]:

        # Producto elemento a elemento: cada columna j se escala por su idf[j]
        tfidf[i, j] = tf[i, j] * idf[j]

In [ ]:
# DF se reinicia en cero por si la celda se ejecuta más de una vez
d_df = cuda.to_device(np.zeros(VOCAB_SIZE, dtype=np.int32))

# Los 4 kernels van en el mismo stream, así que se ejecutan en orden:
# TF y DF dependen del BoW, IDF depende de DF y TF-IDF de TF e IDF
tf_kernel[blocks2d, threads2d](d_bow, d_token_counts, d_tf)
df_kernel[blocks2d, threads2d](d_bow, d_df)
idf_kernel[blocks_vocab, threads](d_df, d_idf, len(corpus))
tfidf_kernel[blocks2d, threads2d](d_tf, d_idf, d_tfidf)

cuda.synchronize()

tfidf_cpu = d_tfidf.copy_to_host()
df_cpu = d_df.copy_to_host()
idf_cpu = d_idf.copy_to_host()

print("TF-IDF no cero primer documento:", np.count_nonzero(tfidf_cpu[0]))
print("DF no cero:", np.count_nonzero(df_cpu))
# Columna más frecuente (stopwords como "de", "la", "que") vs. menos frecuente
print(f"IDF mínimo (término más común): {idf_cpu[df_cpu > 0].min():.3f} | IDF máximo (más raro): {idf_cpu[df_cpu > 0].max():.3f}")

# 15. Bigramas

Un bigrama es una pareja de tokens consecutivos. Ejemplo: `"inteligencia artificial urbana"` produce `["inteligencia artificial", "artificial urbana"]`.


In [ ]:
# Matriz de conteos de bigramas, mismo tamaño que el BoW (hashing a VOCAB_SIZE)
bigram_bow = np.zeros((len(corpus), VOCAB_SIZE), dtype=np.int32)
d_bigram_bow = cuda.to_device(bigram_bow)

# 16. Kernel 7: Conteo de bigramas

In [ ]:
@cuda.jit
def bigram_kernel(token_hashes, token_counts, bigram_bow, vocab_size):

    # Un hilo por documento
    doc_id = cuda.grid(1)

    if doc_id < token_hashes.shape[0]:

        # Con n tokens hay n-1 parejas consecutivas
        for j in range(token_counts[doc_id] - 1):

            # Token actual
            h1 = token_hashes[doc_id, j]

            # Token siguiente
            h2 = token_hashes[doc_id, j + 1]

            # Se combinan los dos hashes; el orden importa:
            # "inteligencia artificial" != "artificial inteligencia"
            # (int() -> aritmética en int64 para evitar desbordamiento)
            bigram_hash = int(h1) * 31 + int(h2)

            # Columna donde cae el bigrama
            bucket = abs(bigram_hash) % vocab_size

            # Suma el conteo del bigrama en la fila del documento
            cuda.atomic.add(bigram_bow, (doc_id, bucket), 1)

In [ ]:
bigram_kernel[blocks_docs, threads](
    d_token_hashes,
    d_token_counts,
    d_bigram_bow,
    VOCAB_SIZE
)

cuda.synchronize()

bigram_cpu = d_bigram_bow.copy_to_host()
print("Bigramas no cero primer documento:", np.count_nonzero(bigram_cpu[0]))
# Debe ser tokens - 1
print("Suma bigramas primer documento:", bigram_cpu[0].sum(), "| tokens - 1:", token_counts_cpu[0] - 1)

# 17. Puntaje de sentimiento léxico

Se usan listas de palabras positivas y negativas. Cada palabra positiva suma +1 y cada negativa resta -1.

Como el corpus ahora es literatura en español, el léxico se amplió con palabras frecuentes en las novelas (*amor*, *alegria*, *muerte*, *dolor*…). Van **sin tilde** porque el corpus se normalizó así.

In [ ]:
# Léxico del notebook original + palabras comunes en las novelas del corpus
positive_words = [
    "mejora", "innovacion", "apoya", "permite", "transforma",
    "ayuda", "anticipar", "futuros", "inteligencia",
    "amor", "alegria", "feliz", "bueno", "buena", "hermosa", "hermoso",
    "bien", "paz", "esperanza", "gloria", "contento", "dulce", "amigo",
]

negative_words = [
    "desafios", "riesgo", "problema", "problemas", "falta",
    "dificultad", "crisis",
    "muerte", "dolor", "triste", "tristeza", "miedo", "odio", "malo",
    "mala", "guerra", "llanto", "desgracia", "enemigo", "muerto",
]

# El kernel compara hashes, no strings: se codifica cada palabra con el
# mismo hash que usa la GPU
pos_hashes = np.array([word_hash(w) for w in positive_words], dtype=np.int32)
neg_hashes = np.array([word_hash(w) for w in negative_words], dtype=np.int32)

# Las listas son pequeñas: se copian una vez a la GPU
d_pos_hashes = cuda.to_device(pos_hashes)
d_neg_hashes = cuda.to_device(neg_hashes)

# Un puntaje entero por documento
sentiment_scores = np.zeros(len(corpus), dtype=np.int32)
d_sentiment_scores = cuda.to_device(sentiment_scores)

print("Hashes positivos:", pos_hashes[:5])
print("Hashes negativos:", neg_hashes[:5])

# 18. Kernel 8: Sentimiento léxico

In [ ]:
@cuda.jit
def sentiment_kernel(token_hashes, token_counts, pos_hashes, neg_hashes, scores):

    # Un hilo por documento
    doc_id = cuda.grid(1)

    if doc_id < token_hashes.shape[0]:

        # Acumulador local (vive en registros, no en memoria global)
        score = 0

        # Recorre los tokens del documento
        for j in range(token_counts[doc_id]):

            h = token_hashes[doc_id, j]

            # Búsqueda lineal en el léxico positivo: +1 si coincide
            for p in range(pos_hashes.size):
                if h == pos_hashes[p]:
                    score += 1

            # Búsqueda lineal en el léxico negativo: -1 si coincide
            for q in range(neg_hashes.size):
                if h == neg_hashes[q]:
                    score -= 1

        # Una sola escritura a memoria global al final
        scores[doc_id] = score

In [ ]:
sentiment_kernel[blocks_docs, threads](
    d_token_hashes,
    d_token_counts,
    d_pos_hashes,
    d_neg_hashes,
    d_sentiment_scores
)

cuda.synchronize()

sentiment_cpu = d_sentiment_scores.copy_to_host()
print("Sentimiento primeros 10 documentos:", sentiment_cpu[:10])
print(f"Documentos positivos: {np.sum(sentiment_cpu > 0):,} | neutros: {np.sum(sentiment_cpu == 0):,} | negativos: {np.sum(sentiment_cpu < 0):,}")

# 19. Mostrar resultados

In [ ]:
# Resumen de los primeros documentos con todas las métricas calculadas en GPU
for i in range(5):
    print("Documento:", corpus[i])
    print("Tokens:", token_counts_cpu[i])
    print("BoW no cero:", np.count_nonzero(bow_cpu[i]))
    print("Bigramas no cero:", np.count_nonzero(bigram_cpu[i]))
    print("TF-IDF no cero:", np.count_nonzero(tfidf_cpu[i]))
    print("Sentimiento:", sentiment_cpu[i])
    print("-" * 80)

# Los documentos más positivos y más negativos del corpus
for etiqueta, idx in (("MÁS POSITIVO", int(np.argmax(sentiment_cpu))),
                      ("MÁS NEGATIVO", int(np.argmin(sentiment_cpu)))):
    print(f"{etiqueta} (doc {idx}, puntaje {sentiment_cpu[idx]}): {corpus[idx]}")

# 20. Versión CPU de referencia

Implementación secuencial en Python + NumPy del mismo pipeline (sin bigramas). Para que la comparación sea justa y la validación sea exacta, aplica **los mismos límites que la GPU**: solo los primeros `MAX_CHARS` caracteres y como máximo `MAX_TOKENS` tokens por documento.

In [ ]:
def tokenize_cpu(doc, max_chars=MAX_CHARS, max_tokens=MAX_TOKENS):
    tokens = []
    current = []
    # Mismo recorte de caracteres que corpus_to_char_matrix
    for c in doc.lower()[:max_chars]:
        # El corpus ya es ASCII, así que isalnum() equivale a [a-z0-9] del kernel
        if c.isalnum():
            current.append(c)
        else:
            # Separador: se cierra la palabra actual
            if current:
                tokens.append("".join(current))
                current = []
    if current:
        tokens.append("".join(current))
    # Mismo tope de tokens que el kernel de tokenización
    return tokens[:max_tokens]


def pipeline_cpu(corpus, vocab_size=2048):
    bow = np.zeros((len(corpus), vocab_size), dtype=np.int32)
    sentiment = np.zeros(len(corpus), dtype=np.int32)

    # Conjuntos para búsqueda O(1) de palabras de sentimiento
    pos_set = set(positive_words)
    neg_set = set(negative_words)

    # Bucle secuencial: un documento a la vez (lo que la GPU hace en paralelo)
    for i, doc in enumerate(corpus):
        tokens = tokenize_cpu(doc)

        for tok in tokens:
            # Mismo hashing trick que el kernel de BoW
            bucket = abs(word_hash(tok)) % vocab_size
            bow[i, bucket] += 1

            if tok in pos_set:
                sentiment[i] += 1
            if tok in neg_set:
                sentiment[i] -= 1

    # TF, DF, IDF y TF-IDF vectorizados con NumPy
    token_counts_local = bow.sum(axis=1)
    tf_local = bow / np.maximum(token_counts_local[:, None], 1)

    df_local = (bow > 0).sum(axis=0)
    idf_local = np.log((1.0 + len(corpus)) / (1.0 + df_local)) + 1.0

    tfidf_local = (tf_local * idf_local).astype(np.float32)

    return bow, tfidf_local, sentiment

# 21. Medición CPU vs GPU

- **CPU**: tiempo de `pipeline_cpu` (tokenización + BoW + TF-IDF + sentimiento).
- **GPU (kernels)**: se mide con eventos CUDA. Incluye la reserva de matrices y los 8 kernels, pero no las copias de entrada/salida.
- **GPU (extremo a extremo)**: además copia la matriz de caracteres a la GPU y trae TF-IDF y sentimiento de vuelta a la CPU. Es la comparación más honesta, porque en un caso real los datos empiezan y terminan en la RAM.

Antes de medir se hace una ejecución de *calentamiento*: la primera llamada a un kernel incluye la compilación JIT de Numba, que no forma parte del cómputo.

In [ ]:
t0 = time.perf_counter()
bow_cpu_ref, tfidf_cpu_ref, sentiment_cpu_ref = pipeline_cpu(corpus, VOCAB_SIZE)
t1 = time.perf_counter()

cpu_time_ms = (t1 - t0) * 1000
print(f"Documentos: {len(corpus):,}")
print(f"Tiempo CPU: {cpu_time_ms:.3f} ms")

In [ ]:
def run_gpu_pipeline(d_chars, d_lengths, n_docs):

    # Configuración de lanzamiento según el número de documentos
    b_docs = (n_docs + threads - 1) // threads
    b2d = ((n_docs + threads2d[0] - 1) // threads2d[0],
           (VOCAB_SIZE + threads2d[1] - 1) // threads2d[1])

    # Buffers en GPU. Los que se llenan con atomic.add deben iniciar en cero;
    # los que se sobrescriben completos solo se reservan
    d_hashes_local = cuda.device_array((n_docs, MAX_TOKENS), dtype=np.int32)
    d_counts_local = cuda.device_array(n_docs, dtype=np.int32)
    d_bow_local = cuda.to_device(np.zeros((n_docs, VOCAB_SIZE), dtype=np.int32))
    d_bigram_local = cuda.to_device(np.zeros((n_docs, VOCAB_SIZE), dtype=np.int32))
    d_df_local = cuda.to_device(np.zeros(VOCAB_SIZE, dtype=np.int32))
    d_tf_local = cuda.device_array((n_docs, VOCAB_SIZE), dtype=np.float32)
    d_idf_local = cuda.device_array(VOCAB_SIZE, dtype=np.float32)
    d_tfidf_local = cuda.device_array((n_docs, VOCAB_SIZE), dtype=np.float32)
    d_sent_local = cuda.device_array(n_docs, dtype=np.int32)

    # Pipeline completo: los 8 kernels en orden de dependencia
    tokenize_kernel[b_docs, threads](d_chars, d_lengths, d_hashes_local, d_counts_local, MAX_TOKENS)
    bow_kernel[b_docs, threads](d_hashes_local, d_counts_local, d_bow_local, VOCAB_SIZE)
    tf_kernel[b2d, threads2d](d_bow_local, d_counts_local, d_tf_local)
    df_kernel[b2d, threads2d](d_bow_local, d_df_local)
    idf_kernel[blocks_vocab, threads](d_df_local, d_idf_local, n_docs)
    tfidf_kernel[b2d, threads2d](d_tf_local, d_idf_local, d_tfidf_local)
    bigram_kernel[b_docs, threads](d_hashes_local, d_counts_local, d_bigram_local, VOCAB_SIZE)
    sentiment_kernel[b_docs, threads](d_hashes_local, d_counts_local, d_pos_hashes, d_neg_hashes, d_sent_local)
    cuda.synchronize()

    return d_bow_local, d_tfidf_local, d_sent_local


def run_gpu_end_to_end(char_matrix, doc_lengths):
    # Igual que run_gpu_pipeline pero incluyendo las copias host <-> device
    d_c = cuda.to_device(char_matrix)
    d_l = cuda.to_device(doc_lengths)
    d_b, d_t, d_s = run_gpu_pipeline(d_c, d_l, char_matrix.shape[0])
    return d_b.copy_to_host(), d_t.copy_to_host(), d_s.copy_to_host()


# Calentamiento (compilación JIT + inicialización del contexto)
run_gpu_pipeline(d_chars, d_lengths, len(corpus))

# --- Solo kernels (datos ya en GPU) ---
start = cuda.event()
end = cuda.event()

start.record()
d_bow_result, d_tfidf_result, d_sent_result = run_gpu_pipeline(d_chars, d_lengths, len(corpus))
end.record()
end.synchronize()

gpu_time_ms = cuda.event_elapsed_time(start, end)

# --- Extremo a extremo (incluye transferencias) ---
t0 = time.perf_counter()
_ = run_gpu_end_to_end(char_matrix, doc_lengths)
gpu_e2e_ms = (time.perf_counter() - t0) * 1000

print(f"Tiempo CPU:                   {cpu_time_ms:10.3f} ms")
print(f"Tiempo GPU (kernels):         {gpu_time_ms:10.3f} ms   speedup {cpu_time_ms / gpu_time_ms:8.2f}x")
print(f"Tiempo GPU (extremo a extremo): {gpu_e2e_ms:8.3f} ms   speedup {cpu_time_ms / gpu_e2e_ms:8.2f}x")

# 22. Validación

Se comparan **todos** los documentos, no solo los primeros 100: el BoW debe ser idéntico, el TF-IDF igual dentro de la tolerancia de `float32` y el sentimiento idéntico. Antes de corregir el hash del kernel 1, esta validación fallaba con el corpus real.

In [ ]:
bow_gpu = d_bow_result.copy_to_host()
tfidf_gpu = d_tfidf_result.copy_to_host()
sent_gpu = d_sent_result.copy_to_host()

print("Sentimiento CPU primeros 10:", sentiment_cpu_ref[:10])
print("Sentimiento GPU primeros 10:", sent_gpu[:10])
print()
print("BoW idéntico en todo el corpus:        ", np.array_equal(bow_cpu_ref, bow_gpu))
print("TF-IDF igual (tolerancia 1e-5):         ", np.allclose(tfidf_cpu_ref, tfidf_gpu, atol=1e-5))
print(f"  Error absoluto máximo TF-IDF:         {np.abs(tfidf_cpu_ref - tfidf_gpu).max():.2e}")
print("Sentimiento idéntico en todo el corpus:", np.array_equal(sentiment_cpu_ref, sent_gpu))
print(f"  Documentos con sentimiento distinto:  {np.sum(sentiment_cpu_ref != sent_gpu)}")

# 23. Escalabilidad: tiempo vs. tamaño del corpus

Para ver cómo crece el beneficio de la GPU con el tamaño del corpus, se repite la medición sobre fracciones crecientes del corpus real (10 %, 25 %, 50 % y 100 %). La GPU solo compensa su costo fijo (lanzamiento de kernels y reserva de memoria) cuando hay suficientes documentos para ocupar todos sus núcleos.

In [ ]:
resultados = []
for frac in (0.10, 0.25, 0.50, 1.00):
    n = max(1, int(len(corpus) * frac))
    sub_corpus = corpus[:n]
    sub_chars = np.ascontiguousarray(char_matrix[:n])
    sub_lengths = np.ascontiguousarray(doc_lengths[:n])

    # CPU
    t0 = time.perf_counter()
    pipeline_cpu(sub_corpus, VOCAB_SIZE)
    t_cpu = (time.perf_counter() - t0) * 1000

    # GPU solo kernels
    d_c, d_l = cuda.to_device(sub_chars), cuda.to_device(sub_lengths)
    ev0, ev1 = cuda.event(), cuda.event()
    ev0.record()
    run_gpu_pipeline(d_c, d_l, n)
    ev1.record()
    ev1.synchronize()
    t_gpu = cuda.event_elapsed_time(ev0, ev1)

    # GPU con transferencias
    t0 = time.perf_counter()
    run_gpu_end_to_end(sub_chars, sub_lengths)
    t_e2e = (time.perf_counter() - t0) * 1000

    resultados.append((n, t_cpu, t_gpu, t_e2e))

print(f"{'Docs':>9} | {'CPU (ms)':>11} | {'GPU kern (ms)':>13} | {'GPU e2e (ms)':>12} | {'Speedup kern':>12} | {'Speedup e2e':>11}")
print("-" * 84)
for n, t_cpu, t_gpu, t_e2e in resultados:
    print(f"{n:>9,} | {t_cpu:>11.1f} | {t_gpu:>13.1f} | {t_e2e:>12.1f} | {t_cpu / t_gpu:>11.1f}x | {t_cpu / t_e2e:>10.1f}x")

# 24. Conclusiones

El flujo completo es:

```text
Texto real (6 libros, >1,2 M palabras) → normalización → caracteres → tokens → Bag of Words → TF-IDF → bigramas → sentimiento
```

**Punto 1.** Todas las celdas de código quedaron comentadas: qué hace cada kernel, cómo se mapea cada hilo a un documento (kernels 1D) o a una celda documento × término (kernels 2D), cuándo hace falta `atomic.add` (DF: muchos hilos escriben la misma posición) y cuándo no es estrictamente necesario (BoW, bigramas: cada hilo escribe solo en su fila).

**Punto 2.** El corpus pasó de 10 frases replicadas a **~60.000 documentos reales** sacados de seis obras en español. Para eso hubo que:

- normalizar tildes y caracteres no ASCII, porque el tokenizador de la GPU solo reconoce `a-z0-9`;
- ajustar `MAX_CHARS`/`MAX_TOKENS` al tamaño de los nuevos documentos;
- **corregir el hash del kernel de tokenización**, que desbordaba y no coincidía con `word_hash` de la CPU. Con el corpus de juguete el error pasaba desapercibido porque solo se validaban 100 documentos; con texto real el sentimiento quedaba mal;
- validar el corpus completo (BoW, TF-IDF y sentimiento) contra la versión CPU.

**Observaciones de rendimiento.**

- Los kernels por documento (tokenización, BoW, bigramas, sentimiento) son *embarrassingly parallel*: cada hilo trabaja en su documento sin comunicarse con los demás.
- Los kernels 2D (TF, DF, TF-IDF) recorren matrices densas `docs × 2048` aunque más del 99 % de las celdas sean cero. Una representación dispersa (CSR) o fusionar TF y TF-IDF en un solo kernel reduciría el tráfico de memoria.
- El *speedup* extremo a extremo es menor que el de solo kernels porque copiar matrices densas por PCIe cuesta. En un pipeline real los datos deberían quedarse en la GPU entre etapas.
- El *hashing trick* con 2048 columnas produce colisiones con un vocabulario real de decenas de miles de palabras. Es un compromiso entre memoria y precisión.

Aunque los modelos modernos de NLP usan embeddings y transformers, este reto sirve para entender las bases computacionales de la representación de texto y del paralelismo en GPU.